# TIFF File Ingest

Ingests TIFF pathology slides into a Unity Catalog Delta table using the `dbx.pixels.Catalog` class.

**Pipeline**
1. Load config from `config.yaml` (source path, pattern, table)
2. Initialise `Catalog` with the target Delta table and UC volume
3. `catalog.catalog()` — recursively discovers all `.tiff` files and enriches each row with path metadata
4. `catalog.save()` — writes the file catalog to the Delta table
5. SQL verification of ingested rows

In [0]:
import sys, pathlib

# Use the editable source tree — no wheel build needed during development
SRC_PATH = "/Workspace/Users/douglas.moore@databricks.com/pixels-tiff/src"

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
    print(f"Added to sys.path: {SRC_PATH}")
else:
    print(f"Already on sys.path: {SRC_PATH}")

In [0]:
%pip install tifffile -q

In [0]:
import yaml, pathlib

CONFIG_PATH = pathlib.Path("/Workspace/Users/douglas.moore@databricks.com/pixels-tiff/notebooks/tiff/config.yaml")

with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

SOURCE_PATH = config["SOURCE_PATH"]
PATTERN     = config["PATTERN"]
TABLE       = config["INDEX"]

# Derive volume from catalog + schema of the index table (dmoore.tiff.<volume>)
_catalog, _schema, _ = TABLE.split(".")
VOLUME = f"{_catalog}.{_schema}.tiff_volume"  # adjust if your volume name differs

WRITE_MODE = "overwrite"  # use 'append' for incremental runs

print(f"SOURCE_PATH : {SOURCE_PATH}")
print(f"PATTERN     : {PATTERN}")
print(f"TABLE       : {TABLE}")
print(f"VOLUME      : {VOLUME}")
print(f"WRITE_MODE  : {WRITE_MODE}")

In [0]:
from dbx.pixels import Catalog

catalog = Catalog(spark, table=TABLE)
print(catalog)

In [0]:
# Recursively discover all TIFF files under SOURCE_PATH.
# .catalog() reads only file metadata (no pixel data is loaded).
catalog_df = catalog.catalog(
    path=SOURCE_PATH,
    pattern=PATTERN,
    recurse=True,
    streaming=False,
)

print(f"Files found: {catalog_df.count()}")
display(catalog_df)

In [0]:
from dbx.pixels.tiff import TiffMetaExtractor

# Enrich the file catalog DataFrame with TIFF metadata.
# TiffMetaExtractor adds a `meta` VARIANT column containing all TIFF tags
# plus derived fields (page_count, is_ome, is_bigtiff, series info, phi_tag_report).
extractor = TiffMetaExtractor(catalog, inputCol="local_path", outputCol="meta")
enriched_df = extractor.transform(catalog_df)

print(f"Schema: {[f.name for f in enriched_df.schema.fields]}")
display(enriched_df.select("local_path", "extension", "meta"))

In [0]:
# Persist the file catalog to the Delta table defined in config.yaml (INDEX).
# Use write_mode='overwrite' for a full refresh, or 'append' for incremental.
catalog.save(enriched_df, mode=WRITE_MODE)

print(f"Saved to: {TABLE} (mode={WRITE_MODE})")

In [0]:
%sql
-- Quick verification: row count and sample paths
SELECT
  COUNT(*)               AS file_count,
  SUM(length) / 1024 / 1024 AS total_size_mb,
  COLLECT_SET(extension) AS extensions
FROM dmoore.tiff.object_catalog

In [0]:
%sql
SELECT
  path,
  length,
  modificationTime,
  extension,
  path_tags,
  file_type,
  meta
FROM dmoore.tiff.object_catalog
ORDER BY modificationTime DESC
LIMIT 50